# FIAP TECH CHALLENGE 1 - GRUPO SSP

## 3. ANÁLISE DE CORRELAÇÃO DOS DADOS

In [23]:
pip install pandas scipy

Note: you may need to restart the kernel to use updated packages.


In [24]:
from pathlib import Path

# Considerando que o Notebook está sendo rodado em /src/notebooks/
BASE_PATH = Path.cwd().resolve().parent.parent
print(f"📁 Base path: {BASE_PATH.absolute()}")

📁 Base path: /Users/matias/Projetos/fiap/tech-challenge-1


In [25]:
import pandas as pd

# import df
df = pd.read_parquet(BASE_PATH / "data" / "processed" / "df_preprocessed.parquet")

# print(df.head())


### 3.1. TESTE QUIQUADRADO

In [26]:
from scipy.stats import chi2_contingency
import pandas as pd

def chi_square_test(df, feature, target="OUT_VEZES"):
    # remover NaN apenas das colunas envolvidas
    temp = df[[feature, target]].dropna()
    
    # tabela de contingência
    contingency = pd.crosstab(temp[feature], temp[target])
    
    chi2, p, dof, expected = chi2_contingency(contingency)
    
    return {
        "feature": feature,
        "chi2": chi2,
        "p_value": p,
        "dof": dof,
        "n": len(temp)
    }


O teste quiquadrado aqui é aplicado somente aos campos que possuem 10 ou menos categorias. Isso pois o teste funciona melhor para categorias maiores e alguns campos possuem muitas categorias como idade, circustância da lesão etc.

In [31]:

candidatas = [
    c for c in df.columns
    if c != "OUT_VEZES" and df[c].nunique(dropna=True) <= 10
]

results = [chi_square_test(df, c) for c in candidatas]
chi_df = pd.DataFrame([r for r in results if "p_value" in r]).sort_values("p_value")
top_chi_df = chi_df.sort_values("chi2", ascending=False).head(10)
print(top_chi_df)

print(top_chi_df['feature'].tolist())

                feature         chi2        p_value  dof      n
30        REL_CONHECIDO  6368.506152   0.000000e+00    1  57022
28  REL_PARCEIRO_INTIMO  5040.725351   0.000000e+00    1  57022
0             AG_AMEACA  2364.861027   0.000000e+00    1  57022
23           VIOL_PSICO  2219.610235   0.000000e+00    1  57022
10           AUTOR_SEXO  1636.839485   0.000000e+00    3  57022
16           SIT_CONJUG   723.671871  2.608316e-155    4  57022
17           VIOL_FINAN   707.550687  6.819776e-156    1  57022
31    REL_INSTITUCIONAL   655.340877  1.540221e-144    1  57022
2              AG_ENFOR   461.558529  2.201556e-102    1  57022
11             CICL_VID   453.963225   6.860487e-96    5  57022
['REL_CONHECIDO', 'REL_PARCEIRO_INTIMO', 'AG_AMEACA', 'VIOL_PSICO', 'AUTOR_SEXO', 'SIT_CONJUG', 'VIOL_FINAN', 'REL_INSTITUCIONAL', 'AG_ENFOR', 'CICL_VID']
